# Template Instantiation Analysis Example

This notebook demonstrates how to use the template analysis functions to understand C++ template instantiation costs in Clang's `-ftime-trace` output.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from trace_analysis import (
    parse_file,
    get_template_instantiation_events,
    get_phase_breakdown,
    get_metadata,
)

import pandas as pd
from datetime import datetime
import plotly.express as px


# Display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## Load Trace File

In [ ]:
# Load your trace file
trace_file = Path(
    "../../../build-trace/library/src/tensor_operation_instance/gpu/conv2d_fwd/CMakeFiles/device_conv2d_fwd_instance.dir/device_conv2d_fwd_xdl_nhwc_kyxc_nhwk_f32_instance.cpp.json"
)
df = parse_file(trace_file)

print(f"Total events: {len(df):,}")
starting_timestamp = datetime.fromtimestamp(df.attrs["beginningOfTime"] / 1e6)
print(f"Starting timestamp: {starting_timestamp.strftime('%Y-%m-%d:%H:%M:%S')}")
print(f"Source file: {df.attrs['sourceFile']}")

In [ ]:
get_metadata(df)

## Compilation Overview

In [ ]:
# Get phase breakdown and display it
breakdown = get_phase_breakdown(df)
print(breakdown)
display(breakdown)

In [ ]:
# Extract data for plotly charts (sunburst, tree-map, or icicle)
plotly_data = breakdown.to_plotly()
fig = px.sunburst(**plotly_data)
fig.show()

## Template Instantiation Analysis

In [ ]:
# Get all template instantiation events (now with parsed columns!)
template_events = get_template_instantiation_events(df)

print(f"Total template instantiation events: {len(template_events):,}")
print(f"Total template time: {template_events['dur'].sum() / 1000:.1f} ms")
display(template_events)

### Examine Parsed Columns

The `get_template_instantiation_events()` function automatically parses the `arg_detail` column into structured fields:

In [ ]:
# Show the new parsed columns
print("Parsed columns available:")
print("- namespace: Top-level namespace (e.g., 'std', 'ck')")
print("- template_name: Template name without parameters")
print("- full_qualified_name: Full namespace::template_name")
print("- param_count: Number of template parameters")
print("- is_ck_type: Boolean indicating CK library types")
print("- is_nested: Boolean indicating nested templates")
print()

# Display sample of parsed data
template_events[
    ["namespace", "template_name", "param_count", "is_ck_type", "is_nested", "dur"]
].head(20)

### Analysis by Namespace

In [ ]:
# Group by namespace to see where time is spent
namespace_summary = (
    template_events.groupby("namespace")
    .agg({"dur": ["count", "sum", "mean"], "param_count": "mean"})
    .round(2)
)

namespace_summary.columns = ["count", "total_dur", "avg_dur", "avg_params"]
namespace_summary["total_ms"] = namespace_summary["total_dur"] / 1000
namespace_summary = namespace_summary.sort_values("total_dur", ascending=False)

print("\nTemplate Instantiation Time by Namespace:")
display(namespace_summary)

### CK Library Templates Analysis

In [ ]:
# Filter to CK types only
ck_templates = template_events[template_events["is_ck_type"]].copy()

print(f"CK template instantiations: {len(ck_templates):,}")
print(f"CK template time: {ck_templates['dur'].sum() / 1000:.1f} ms")
print(
    f"Percentage of total template time: {100 * ck_templates['dur'].sum() / template_events['dur'].sum():.1f}%"
)
print()

# Top CK templates by time
ck_by_name = (
    ck_templates.groupby("template_name")
    .agg({"dur": ["count", "sum", "mean"]})
    .round(2)
)
ck_by_name.columns = ["count", "total_dur", "avg_dur"]
ck_by_name["total_ms"] = ck_by_name["total_dur"] / 1000
ck_by_name = ck_by_name.sort_values("total_dur", ascending=False)

print("\nTop CK Templates by Total Time:")
display(ck_by_name.head(20))